# 🔬 Material Science Regression Analysis
## Predicting Compressive Strength After Treatment Using ML Models

**Features:** Fly Ash, SF (Silica Fume), LP (Lime Powder), CW Before  
**Target:** CW After (Compressive Strength After Treatment)  
**Models:** Linear Regression · Random Forest · KNN · LightGBM · XGBoost

---

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & DATA LOADING
# =============================================================================
# Description: Import all necessary libraries and load the dataset.
#              Perform initial exploration: shape, dtypes, missing values,
#              and descriptive statistics.
# =============================================================================

# --- Core Libraries ---
import pandas as pd
import numpy as np

# --- Visualization Libraries ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# --- Scikit-Learn: Preprocessing & Metrics ---
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Scikit-Learn: Models ---
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

# --- Gradient Boosting Libraries ---
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

# --- Utility ---
import warnings
warnings.filterwarnings('ignore')          # Suppress non-critical warnings

# Set global random seed for full reproducibility across all experiments
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Plot Aesthetics (Publication-Ready Style) ---
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({
    'figure.dpi'      : 150,
    'axes.titleweight': 'bold',
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
})

# =============================================================================
# DATA LOADING
# Upload your CSV file to Colab using the Files panel (folder icon on the left),
# then update the FILE_PATH variable below.
# =============================================================================

FILE_PATH = 'your_dataset.csv'            # <-- UPDATE THIS PATH

# Define the exact column names as they appear in the CSV
FEATURE_COLS = ['Fly ash', 'SF', 'LP', 'CW before']
TARGET_COL   = 'CW after'

# Load the dataset into a pandas DataFrame
df_raw = pd.read_csv(FILE_PATH)

print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'  Rows    : {df_raw.shape[0]}')
print(f'  Columns : {df_raw.shape[1]}')
print(f'  Features: {FEATURE_COLS}')
print(f'  Target  : {TARGET_COL}')
print('=' * 55)

# --- Missing Value Audit ---
print('\n📋 Missing Values per Column:')
missing_summary = df_raw.isnull().sum()
print(missing_summary[missing_summary >= 0].to_string())  # Show all columns

total_missing = df_raw.isnull().sum().sum()
if total_missing == 0:
    print('\n✅ No missing values detected. Dataset is clean.')
else:
    print(f'\n⚠️  Total missing values: {total_missing}. Handle before proceeding.')

# --- Descriptive Statistics ---
print('\n📊 Descriptive Statistics:')
display(df_raw[FEATURE_COLS + [TARGET_COL]].describe().round(4))

# --- Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(7, 5))
corr_matrix = df_raw[FEATURE_COLS + [TARGET_COL]].corr()
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature–Target Correlation Matrix')
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('💾 Saved: fig_correlation_heatmap.png')

In [ ]:
# =============================================================================
# CELL 2: DATA PREPROCESSING
# =============================================================================
# Description: Separate features (X) and target (y), perform an 80/20
#              train-test split, and apply StandardScaler to the features.
#              Scaling is essential for distance-based (KNN) and
#              gradient-based (Linear Regression) algorithms to function
#              correctly without magnitude bias.
# =============================================================================

# --- Separate Features and Target ---
X = df_raw[FEATURE_COLS].values    # Feature matrix  (n_samples × n_features)
y = df_raw[TARGET_COL].values      # Target vector   (n_samples,)

print(f'Feature matrix shape : {X.shape}   (samples × features)')
print(f'Target vector shape  : {y.shape}   (samples,)')

# --- Train / Test Split (80% / 20%) ---
# shuffle=True ensures class distribution is maintained across the split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = RANDOM_STATE,
    shuffle      = True
)

print(f'\nTrain samples : {X_train.shape[0]}  ({100 * X_train.shape[0] / X.shape[0]:.0f}%)')
print(f'Test  samples : {X_test.shape[0]}   ({100 * X_test.shape[0]  / X.shape[0]:.0f}%)')

# --- Feature Scaling: StandardScaler ---
# Fit ONLY on training data to prevent data leakage from test set.
# Then transform both train and test using the same learned parameters
# (mean and standard deviation from training).
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # Fit + Transform on train
X_test_scaled  = scaler.transform(X_test)         # Transform only on test

print('\n✅ StandardScaler applied successfully.')
print(f'   Scaler means (train) : {scaler.mean_.round(4)}')
print(f'   Scaler std   (train) : {scaler.scale_.round(4)}')
print('\n📌 Note: Tree-based models (RF, LGB, XGB) are scale-invariant,')
print('         but using scaled data here ensures a unified pipeline.')

In [ ]:
# =============================================================================
# CELL 3: MODEL TRAINING & EVALUATION
# =============================================================================
# Description: Define, train, and evaluate all five regression models on
#              the held-out test set. Compute MAE, RMSE, and R² for each
#              model and present a ranked comparison table.
#
# Models:
#   1. Linear Regression  – Baseline parametric model
#   2. Random Forest      – Ensemble of decision trees (bagging)
#   3. K-Nearest Neighbors– Non-parametric, distance-based
#   4. LightGBM           – Gradient boosting with leaf-wise growth
#   5. XGBoost            – Gradient boosting with depth-wise growth
# =============================================================================

# --- Helper Function: Compute Regression Metrics ---
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Compute MAE, RMSE, and R² for a set of predictions.

    Parameters
    ----------
    y_true : array-like of actual target values
    y_pred : array-like of model-predicted values

    Returns
    -------
    dict with keys 'MAE', 'RMSE', 'R2'
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}


# --- Model Registry ---
# Each entry: (model_name, model_instance)
# All hyperparameters are set to well-calibrated defaults for materials data.
model_registry = {
    'Linear Regression': LinearRegression(),

    'Random Forest': RandomForestRegressor(
        n_estimators = 300,        # Number of trees in the forest
        max_depth    = None,       # Trees grow until leaves are pure
        min_samples_split = 2,
        random_state = RANDOM_STATE,
        n_jobs       = -1          # Use all available CPU cores
    ),

    'K-Nearest Neighbors': KNeighborsRegressor(
        n_neighbors = 5,           # k = 5 is a robust default
        weights     = 'distance',  # Closer neighbors have higher influence
        metric      = 'euclidean'
    ),

    'LightGBM': LGBMRegressor(
        n_estimators   = 500,
        learning_rate  = 0.05,
        num_leaves     = 31,       # Controls model complexity
        random_state   = RANDOM_STATE,
        n_jobs         = -1,
        verbose        = -1        # Suppress training logs
    ),

    'XGBoost': XGBRegressor(
        n_estimators   = 500,
        learning_rate  = 0.05,
        max_depth      = 6,
        subsample      = 0.8,      # Row subsampling to prevent overfitting
        colsample_bytree = 0.8,   # Column subsampling per tree
        random_state   = RANDOM_STATE,
        n_jobs         = -1,
        verbosity      = 0
    )
}

# --- Training Loop ---
results_list   = []     # Stores metric dicts for each model
trained_models = {}     # Stores fitted model objects for later use
predictions    = {}     # Stores test-set predictions for each model

print('Training models...\n')
for model_name, model in model_registry.items():
    # Fit the model on scaled training data
    model.fit(X_train_scaled, y_train)

    # Generate predictions on the held-out scaled test set
    y_pred = model.predict(X_test_scaled)

    # Compute evaluation metrics
    metrics = compute_metrics(y_test, y_pred)
    metrics['Model'] = model_name

    # Store results
    results_list.append(metrics)
    trained_models[model_name] = model
    predictions[model_name]    = y_pred

    print(f'  ✅ {model_name:<25} | MAE: {metrics["MAE"]:6.4f}  '
          f'RMSE: {metrics["RMSE"]:6.4f}  R²: {metrics["R2"]:6.4f}')

# --- Build Comparison DataFrame ---
df_results = (
    pd.DataFrame(results_list)
      .set_index('Model')
      [['MAE', 'RMSE', 'R2']]      # Column order for clarity
      .sort_values('R2', ascending=False)  # Rank by R² (higher is better)
      .round(4)
)

print('\n' + '=' * 55)
print('  MODEL PERFORMANCE COMPARISON (Test Set)')
print('=' * 55)
display(df_results.style
    .background_gradient(subset=['R2'],  cmap='Greens')
    .background_gradient(subset=['MAE'], cmap='Reds_r')
    .background_gradient(subset=['RMSE'],cmap='Reds_r')
    .format(precision=4)
)

# Identify the best model by R²
BEST_MODEL_NAME = df_results['R2'].idxmax()
print(f'\n🏆 Best Model: {BEST_MODEL_NAME}  (R² = {df_results.loc[BEST_MODEL_NAME, "R2"]:.4f})')

In [ ]:
# =============================================================================
# CELL 4: ROBUSTNESS CHECK — 10-FOLD CROSS-VALIDATION
# =============================================================================
# Description: Perform stratified K-Fold Cross-Validation (k=10) on the
#              top-performing models to assess generalizability and stability.
#              A high Mean R² with a low Standard Deviation confirms the
#              model is not overfitted to a specific train/test split.
#
# NOTE: Cross-validation is run on the FULL scaled dataset (X_scaled)
#       to provide the most reliable stability estimate.
# =============================================================================

# Scale the entire feature matrix for cross-validation
X_full_scaled = scaler.fit_transform(X)    # Re-fit on entire dataset for CV

# --- K-Fold Configuration ---
N_SPLITS   = 10         # 10-fold cross-validation (standard in ML literature)
CV_SCORING = 'r2'       # Metric used for cross-validation scoring

kf = KFold(
    n_splits    = N_SPLITS,
    shuffle     = True,
    random_state= RANDOM_STATE
)

# --- Select Top Models for CV (all 5 in this workflow) ---
TOP_MODELS_FOR_CV = list(model_registry.keys())

cv_results = []

print(f'Running {N_SPLITS}-Fold Cross-Validation on {len(TOP_MODELS_FOR_CV)} models...\n')

for model_name in TOP_MODELS_FOR_CV:
    model = model_registry[model_name]     # Use original (unfitted) instance

    # cross_val_score returns an array of R² scores, one per fold
    fold_scores = cross_val_score(
        estimator = model,
        X         = X_full_scaled,
        y         = y,
        cv        = kf,
        scoring   = CV_SCORING,
        n_jobs    = -1
    )

    mean_r2 = fold_scores.mean()
    std_r2  = fold_scores.std()

    cv_results.append({
        'Model'       : model_name,
        'Mean R²'     : round(mean_r2, 4),
        'Std Dev R²'  : round(std_r2, 4),
        'Min Fold R²' : round(fold_scores.min(), 4),
        'Max Fold R²' : round(fold_scores.max(), 4),
    })

    stability = '✅ Stable' if std_r2 < 0.05 else '⚠️  Variable'
    print(f'  {model_name:<25} | Mean R²: {mean_r2:.4f}  ± {std_r2:.4f}  {stability}')

# --- Cross-Validation Summary Table ---
df_cv = (
    pd.DataFrame(cv_results)
      .set_index('Model')
      .sort_values('Mean R²', ascending=False)
)

print('\n' + '=' * 60)
print(f'  {N_SPLITS}-FOLD CROSS-VALIDATION RESULTS')
print('=' * 60)
display(df_cv.style
    .background_gradient(subset=['Mean R²'], cmap='Greens')
    .background_gradient(subset=['Std Dev R²'], cmap='YlOrRd_r')
    .format(precision=4)
)

print('\n📌 Interpretation: A low Std Dev R² (< 0.05) indicates')
print('   the model performs consistently across data subsets — key')
print('   evidence of robustness in engineering paper reporting.')

In [ ]:
# =============================================================================
# CELL 5: RESIDUAL ANALYSIS — STATISTICAL SOUNDNESS
# =============================================================================
# Description: For the best-performing model, perform a residual analysis
#              to validate the statistical assumptions of regression:
#
#   1. Residual vs. Fitted Plot   → Check for heteroscedasticity / bias
#   2. Residual Distribution Plot → Check for normality of errors
#   3. Q-Q Plot                   → Assess normality graphically
#
# A good model exhibits: randomly scattered residuals around zero,
# no systematic patterns, and approximately normally distributed errors.
# =============================================================================

from scipy import stats    # For Q-Q plot and Shapiro-Wilk normality test

# Retrieve the best model's test-set predictions
y_pred_best = predictions[BEST_MODEL_NAME]

# Compute residuals: e_i = y_actual - y_predicted
residuals = y_test - y_pred_best

print(f'Residual Analysis for: {BEST_MODEL_NAME}')
print(f'  Mean of Residuals   : {residuals.mean():.6f}  (ideal: ≈ 0.0000)')
print(f'  Std  of Residuals   : {residuals.std():.4f}')
print(f'  Min Residual        : {residuals.min():.4f}')
print(f'  Max Residual        : {residuals.max():.4f}')

# --- Shapiro-Wilk Normality Test ---
stat_sw, p_sw = stats.shapiro(residuals)
print(f'\n  Shapiro-Wilk Stat   : {stat_sw:.4f}')
print(f'  Shapiro-Wilk p-val  : {p_sw:.4f}', end='  ')
print('(Residuals appear normal ✅)' if p_sw > 0.05 else '(Residuals non-normal ⚠️)')

# =============================================================================
# FIGURE: Three-panel Residual Diagnostic Plot
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f'Residual Diagnostic Plots — {BEST_MODEL_NAME}',
    fontsize=14, fontweight='bold', y=1.02
)

# --- Panel 1: Residuals vs. Fitted Values ---
ax1 = axes[0]
ax1.scatter(y_pred_best, residuals,
            color='steelblue', alpha=0.7, edgecolors='white', linewidths=0.4, s=60)
ax1.axhline(y=0, color='crimson', linestyle='--', linewidth=1.5, label='Zero line')
ax1.set_xlabel('Fitted Values (Predicted CW After)')
ax1.set_ylabel('Residuals (Actual − Predicted)')
ax1.set_title('Residuals vs. Fitted Values')
ax1.legend(fontsize=9)

# --- Panel 2: Residual Distribution (Histogram + KDE) ---
ax2 = axes[1]
sns.histplot(residuals, kde=True, ax=ax2,
             color='steelblue', edgecolor='white',
             line_kws={'color': 'crimson', 'linewidth': 2})
ax2.axvline(x=0, color='crimson', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Residual Value')
ax2.set_ylabel('Frequency')
ax2.set_title('Residual Distribution')

# --- Panel 3: Q-Q Plot (Quantile-Quantile) ---
ax3 = axes[2]
qq_data = stats.probplot(residuals, dist='norm')
ax3.scatter(qq_data[0][0], qq_data[0][1],
            color='steelblue', alpha=0.7, edgecolors='white', linewidths=0.4, s=60)
ax3.plot(qq_data[0][0],
         qq_data[1][1] + qq_data[1][0] * qq_data[0][0],
         color='crimson', linewidth=1.8, linestyle='--', label='Normal reference')
ax3.set_xlabel('Theoretical Quantiles')
ax3.set_ylabel('Sample Quantiles')
ax3.set_title('Q-Q Plot of Residuals')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_residual_analysis.png', bbox_inches='tight')
plt.show()
print('💾 Saved: fig_residual_analysis.png')

In [ ]:
# =============================================================================
# CELL 6: PUBLICATION-QUALITY VISUALIZATIONS
# =============================================================================
# Description: Generate three key figures suitable for inclusion in an
#              engineering research paper (CEP / FYP report):
#
#   Figure 1: R² Bar Chart          — Cross-model performance comparison
#   Figure 2: Feature Importance    — Variable influence from XGBoost & RF
#   Figure 3: Actual vs. Predicted  — Model accuracy scatter with 45° line
# =============================================================================

# Professional color palette (colorblind-accessible)
MODEL_COLORS = ['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51']
ACCENT_COLOR  = '#e63946'     # Reference line color
PRIMARY_COLOR = '#2a9d8f'     # Primary highlight color

model_names_ordered = df_results.index.tolist()   # Sorted by R² descending
r2_values_ordered   = df_results['R2'].tolist()

# =============================================================================
# FIGURE 1: R² Comparison Bar Chart
# =============================================================================
fig1, ax1 = plt.subplots(figsize=(10, 5))

bars = ax1.barh(
    model_names_ordered, r2_values_ordered,
    color   = MODEL_COLORS[:len(model_names_ordered)],
    edgecolor='white', height=0.55
)

# Annotate each bar with its R² value
for bar, r2 in zip(bars, r2_values_ordered):
    ax1.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f'{r2:.4f}', va='center', ha='left', fontsize=10, fontweight='bold'
    )

ax1.set_xlabel('R² Score (Coefficient of Determination)', fontsize=11)
ax1.set_title('Model Performance Comparison — R² Score (Test Set)', fontsize=13, fontweight='bold')
ax1.set_xlim(0, 1.12)
ax1.axvline(x=1.0, color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax1.invert_yaxis()     # Best model at top
ax1.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig1_r2_comparison.png', bbox_inches='tight', dpi=300)
plt.show()
print('💾 Saved: fig1_r2_comparison.png')

# =============================================================================
# FIGURE 2: Feature Importance (XGBoost + Random Forest Averaged)
# =============================================================================
xgb_model = trained_models['XGBoost']
rf_model  = trained_models['Random Forest']

# Retrieve normalized feature importance scores
xgb_importance = xgb_model.feature_importances_
rf_importance  = rf_model.feature_importances_

# Ensemble average of both importance vectors
avg_importance = (xgb_importance + rf_importance) / 2

# Sort features by average importance (descending)
importance_df = (
    pd.DataFrame({
        'Feature'         : FEATURE_COLS,
        'XGBoost'         : xgb_importance,
        'Random Forest'   : rf_importance,
        'Avg Importance'  : avg_importance
    })
    .sort_values('Avg Importance', ascending=True)  # ascending for horizontal bar
)

fig2, ax2 = plt.subplots(figsize=(9, 4))

x_idx = np.arange(len(importance_df))
bar_w = 0.28

ax2.barh(x_idx + bar_w, importance_df['XGBoost'],        height=bar_w, label='XGBoost',       color='#264653')
ax2.barh(x_idx,          importance_df['Random Forest'], height=bar_w, label='Random Forest', color='#2a9d8f')
ax2.barh(x_idx - bar_w, importance_df['Avg Importance'], height=bar_w, label='Average',        color='#e9c46a')

ax2.set_yticks(x_idx)
ax2.set_yticklabels(importance_df['Feature'], fontsize=11)
ax2.set_xlabel('Normalized Feature Importance Score', fontsize=11)
ax2.set_title('Feature Importance — XGBoost vs. Random Forest', fontsize=13, fontweight='bold')
ax2.legend(loc='lower right', fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig2_feature_importance.png', bbox_inches='tight', dpi=300)
plt.show()
print('💾 Saved: fig2_feature_importance.png')
display(importance_df.set_index('Feature').round(4))

# =============================================================================
# FIGURE 3: Actual vs. Predicted Scatter Plot (Best Model)
# =============================================================================
fig3, ax3 = plt.subplots(figsize=(7, 6))

# Perfect prediction line: y = x (45°)
all_vals    = np.concatenate([y_test, y_pred_best])
axis_min    = all_vals.min() * 0.95
axis_max    = all_vals.max() * 1.05
perfect_line= np.linspace(axis_min, axis_max, 100)

ax3.scatter(
    y_test, y_pred_best,
    color=PRIMARY_COLOR, alpha=0.75, edgecolors='white',
    linewidths=0.5, s=75, zorder=3, label='Data points'
)
ax3.plot(
    perfect_line, perfect_line,
    color=ACCENT_COLOR, linestyle='--', linewidth=2,
    label='Perfect prediction (y = x)', zorder=2
)

# Annotate with test-set metrics
test_r2  = df_results.loc[BEST_MODEL_NAME, 'R2']
test_mae = df_results.loc[BEST_MODEL_NAME, 'MAE']
ax3.text(
    0.05, 0.92,
    f'R² = {test_r2:.4f}\nMAE = {test_mae:.4f}',
    transform=ax3.transAxes,
    fontsize=11, verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8, edgecolor='gray')
)

ax3.set_xlabel('Actual CW After (Measured)', fontsize=11)
ax3.set_ylabel('Predicted CW After (Model Output)', fontsize=11)
ax3.set_title(
    f'Actual vs. Predicted — {BEST_MODEL_NAME}',
    fontsize=13, fontweight='bold'
)
ax3.set_xlim(axis_min, axis_max)
ax3.set_ylim(axis_min, axis_max)
ax3.legend(fontsize=9)
ax3.spines[['top', 'right']].set_visible(False)
ax3.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.savefig('fig3_actual_vs_predicted.png', bbox_inches='tight', dpi=300)
plt.show()
print('💾 Saved: fig3_actual_vs_predicted.png')

# =============================================================================
# SUMMARY REPORT
# =============================================================================
print('\n' + '=' * 60)
print('  FINAL SUMMARY REPORT')
print('=' * 60)
print(f'  Best Model    : {BEST_MODEL_NAME}')
print(f'  Test R²       : {df_results.loc[BEST_MODEL_NAME, "R2"]:.4f}')
print(f'  Test MAE      : {df_results.loc[BEST_MODEL_NAME, "MAE"]:.4f}')
print(f'  Test RMSE     : {df_results.loc[BEST_MODEL_NAME, "RMSE"]:.4f}')
print(f'  CV Mean R²    : {df_cv.loc[BEST_MODEL_NAME, "Mean R²"]:.4f}')
print(f'  CV Std Dev R² : {df_cv.loc[BEST_MODEL_NAME, "Std Dev R²"]:.4f}')
print('\n  Figures saved:')
print('    fig_correlation_heatmap.png')
print('    fig_residual_analysis.png')
print('    fig1_r2_comparison.png')
print('    fig2_feature_importance.png')
print('    fig3_actual_vs_predicted.png')
print('=' * 60)